## Futurae Auth Pipeline

In [0]:
# Import Libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json

### Bronze Layer

In [0]:
# Load data from volume
df = spark.read.json('/Volumes/workspace/default/bronze_raw/futurae_auth_events_data.json')


In [0]:
# View data
display(df)

In [0]:
df.show()

In [0]:
# Total rows
df.count()


In [0]:
# Total columns
df.columns

In [0]:
# Print schema
df.printSchema()

### .Select and .Filter

#### .Select is used to pick specific columns from data

In [0]:
df.select('event_id', 'event_type', 'timestamp', 'result').show(10)

In [0]:
# Using display command for Databricks to view the data
# select
display(df.select('event_id', 'event_type', 'timestamp','result'))
# df.select('event_id', 'event_type', 'timestamp',).display()

In [0]:
df.select('location.country', 'device_info.os', 'user_id').display()

#### Using .Filter

In [0]:
# Authenticating events that failed
df.filter(df.result=='failure').display()

In [0]:
# Filtering failed results with hihg risk score
risky_failed_login = df.filter(
(df.result=='failure') & 
(df.risk_score > 0.7)
)

risky_failed_login.display()

In [0]:
# Combining the filter and select
risky_failed_login.select('user_id', 'result', 'risk_score').display()

#### Save as delta format

In [0]:
# Save as table
df.write.format("delta").mode("overwrite").saveAsTable('workspace.default.bronze_auth_events')

In [0]:
# Save as file
df.write.format('delta').mode('overwrite').save('/Volumes/workspace/default/bronze_raw/bronze_auth_events')

### Silver Layer
##### Cleaning Data Phase

In [0]:
# Calling bronze table
bronze_df = spark.table('workspace.default.bronze_auth_events')
display(bronze_df)

#### Data Quality Check

In [0]:
# Check 1 - Duplicates
bronze_df.count()-bronze_df.distinct().count()

In [0]:
# Check 2 - Null values
bronze_df.filter(col('user_id').isNull()).count()

#### Data Cleaning And Flattening
##### Data Cleaning

In [0]:
# Handling Duplicatea
bronze_df = bronze_df.dropDuplicates()

In [0]:
# Handling Null values
# By droping Null values
bronze_df = bronze_df.na.drop(subset=['user_id', 'event_id'])
# By filling Null values
bronze_df = bronze_df.na.fill({'risk_score': 0.0})
# By filtering Null values
bronze_df = bronze_df.filter(col('user_id').isNotNull() & col('event_id').isNotNull())

##### Flatten Important Columns For Analysis

In [0]:
# Select and flatten key columns
silver_df = bronze_df.select(
    'user_id',
    'event_id',
    'timestamp',
    'auth_method',
    'risk_score',
    'result',
    col('location.country').alias('country'),
    col('location.city').alias('city'),
    col('device_info.device_type').alias('device_type'),
    col('network_info.isp').alias('network_isp'),
)

silver_df.display()                        

In [0]:
# Save as table
silver_df.write.format('delta').mode('overwrite').saveAsTable('workspace.default.silver_auth_events')

### Gold Layer

In [0]:
gold_df = spark.table('workspace.default.silver_auth_events')

In [0]:
# Number of events for each country
country_counts = gold_df.groupBy('country').count().orderBy('count',ascending=False)

country_counts.display()

### Spark Sql

In [0]:
%sql
SELECT country, count(*) as events_count
FROM silver_auth_events
GROUP BY country
ORDER BY events_count DESC
LIMIT 10

In [0]:
spark.sql('SELECT country, count(*) as events_count FROM silver_auth_events GROUP BY country ORDER BY events_count DESC LIMIT 10').display()

In [0]:
# country_counts as a table in  gold layer
country_counts.write.format('delta').mode('overwrite').saveAsTable('workspace.default.country_counts')

In [0]:
.